<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/hands_on_ml_with_scikit-learn_Aurelien_textbook/Chap_16_NLP_with_RNNs_and_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf

In [ ]:
shakespeare_url = "https://homl.info/shakespeare"
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)


1115394/1115394 [==============================] - 0s 0us/step


In [ ]:
with open(filepath) as f:
  shakespeare_text = f.read()

In [ ]:
print(shakespeare_text[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [ ]:
# encode the text using TextVectorization
text_vec_layer = tf.keras.layers.TextVectorization(split="character")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]

In [ ]:
encoded

<tf.Tensor: shape=(1060997,), dtype=int64, numpy=array([20,  7, 10, ..., 11, 21, 12])>

Each character is now mapped to an integer, starting at 2. The TextVectorization layer reserved value 0 for padding tokens and it reserved 1 for unknown characters. Since we won't be needing either of these tokens for now, let's subtract 2 from the character IDs, and compute the number of distinct characters and the total number of characters.

In [ ]:
encoded -=  2
n_tokens = text_vec_layer.vocabulary_size() - 2
dataset_size = len(encoded)

In [ ]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
  ds = tf.data.Dataset.from_tensor_slices(sequence)
  ds = ds.window(length + 1, shift=1, drop_remainder=True)
  ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
  if shuffle:
    ds = ds.shuffle(buffer_size = 100_000, seed=seed)
  ds = ds.batch(batch_size)
  return  ds.map(lambda window: (window[:, : -1], window[:, 1:])).prefetch(1)

In [ ]:
length = 100
tf.random.set_seed(42)
train_set = to_dataset(encoded[:1_000_000],
                       length, shuffle=True, seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], length)
test_set = to_dataset(encoded[1_060_000:], length)

The length above was set to 100 but one can try tuning it: it's easier and faster to train RNNs on shorter input sequences, but the RNN will not be able to learn any pattern longer than the set length so we shouldn't make it too small

In [ ]:
# Lets build and train a model with one GRU layer composed of 128 units
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "my_shakespeare_model", monitor="val_accuracy", save_best_only=True
)
history = model.fit(train_set, validation_data=valid_set, epochs=10,
                    callbacks=[model_ckpt])

Epoch 1/10
  31247/Unknown - 440s 13ms/step - loss: 1.3710 - accuracy: 0.5713

31247/31247 [==============================] - 460s 14ms/step - loss: 1.3710 - accuracy: 0.5713 - val_loss: 1.6654 - val_accuracy: 0.5101
Epoch 2/10
31244/31247 [============================>.] - ETA: 0s - loss: 1.2782 - accuracy: 0.5942

31247/31247 [==============================] - 407s 12ms/step - loss: 1.2782 - accuracy: 0.5942 - val_loss: 1.6604 - val_accuracy: 0.5128
Epoch 3/10
31246/31247 [============================>.] - ETA: 0s - loss: 1.2608 - accuracy: 0.5980

31247/31247 [==============================] - 392s 12ms/step - loss: 1.2608 - accuracy: 0.5980 - val_loss: 1.6638 - val_accuracy: 0.5169
Epoch 4/10
31247/31247 [==============================] - 395s 12ms/step - loss: 1.2526 - accuracy: 0.5999 - val_loss: 1.6569 - val_accuracy: 0.5169
Epoch 5/10
31247/31247 [==============================] - ETA: 0s - loss: 1.2463 - accuracy: 0.6010

31247/31247 [==============================] - 404s 12ms/step - loss: 1.2463 - accuracy: 0.6010 - val_loss: 1.6507 - val_accuracy: 0.5190
Epoch 6/10
31246/31247 [============================>.] - ETA: 0s - loss: 1.2430 - accuracy: 0.6018

31247/31247 [==============================] - 417s 13ms/step - loss: 1.2430 - accuracy: 0.6018 - val_loss: 1.6508 - val_accuracy: 0.5200
Epoch 7/10
31247/31247 [==============================] - ETA: 0s - loss: 1.2396 - accuracy: 0.6028

31247/31247 [==============================] - 418s 13ms/step - loss: 1.2396 - accuracy: 0.6028 - val_loss: 1.6615 - val_accuracy: 0.5237
Epoch 8/10
31243/31247 [============================>.] - ETA: 0s - loss: 1.2365 - accuracy: 0.6035

31247/31247 [==============================] - 400s 12ms/step - loss: 1.2365 - accuracy: 0.6035 - val_loss: 1.6481 - val_accuracy: 0.5238
Epoch 9/10
31242/31247 [============================>.] - ETA: 0s - loss: 1.2347 - accuracy: 0.6039

31247/31247 [==============================] - 410s 13ms/step - loss: 1.2347 - accuracy: 0.6040 - val_loss: 1.6531 - val_accuracy: 0.5257
Epoch 10/10
31247/31247 [==============================] - 410s 13ms/step - loss: 1.2336 - accuracy: 0.6041 - val_loss: 1.6586 - val_accuracy: 0.5232


The above model does not handle text preprocessing, so we'll wrap it in a final model containing the tf.keras.layers.TextVectorization layer as the first layer, plus a tf.keras.layers.Lambda layer to subtract 2 from the character IDs since we're not using the padding and unknown tokens for now.

In [ ]:
!pip install Path

In [ ]:
from pathlib import Path

In [ ]:
# shakespeare_model = tf.keras.Sequential([
#     text_vec_layer,
#     tf.keras.layers.Lambda(lambda x: x - 2),
#     model
# ])

url = "https://github.com/ageron/data/raw/main/shakespeare_model.tgz"
path = tf.keras.utils.get_file("shakespeare_model.tgz", url, extract=True)
model_path = Path(path).with_name("shakespeare_model")
shakespeare_model = tf.keras.models.load_model(model_path)

In [ ]:
# predict the next character in a sentence
n_chars = 50
text = "Well, this is stupid and disappointin"
for _ in range(50):
  y_proba = shakespeare_model.predict([text])[0, -1]
  y_pred = tf.argmax(y_proba)
  c = text_vec_layer.get_vocabulary()[y_pred + 2]
  text += c

1/1 [==============================] - 0s 57ms/step


In [ ]:
print(text)

Well, this is stupid and disappointinble kind
than the dujefore to the deathcecisecimen


### Generating Fake Shakespearean Text
To generate new text using the Char-RNN model, we could feed it some text, make the model predict the most likely next letter, add it at the end of the text, then give the extended text to the model to guess the next letter and so on. This is called greedy decoding. In practise, this often leans to the same words being repeated over and over again. Instead, we can sample the next character randomly, with a probability equal to the estimated probability, using tf.random.categorical(). This will generate more diverse and interesting text. The categorical() function samples random class indices, given the class log probabilities(logits). For example

In [ ]:
log_probas = tf.math.log([[0.5, 0.4, 0.1]])
tf.random.set_seed(42)
tf.random.categorical(log_probas, num_samples=8)

To have more control over the diversity of the generated text, we can divide the logits by a number called the temperature, which we can tweak as we wish. A temperature close to 0 favours high probability characters, while a high temperature gives all characters an equal probability. Lower temperatures are typically preferred when generating fairly rigid and precise text, such as mathematical equations, while higher temperatures are preferred when generating more diverse and creative text. The function below uses this approach to pick the next character to add to the input text

In [ ]:
def next_char(text, temperature=1):
  y_proba = shakespeare_model.predict([text])[0, -1:]
  rescaled_logits = tf.math.log(y_proba) / temperature
  char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
  return text_vec_layer.get_vocabulary()[char_id + 2]

In [ ]:
def extend_text(text, n_chars=50, temperature=1):
  for _ in range(n_chars):
    text += next_char(text, temperature)
  return text

In [ ]:
tf.random.set_seed(42)
print(extend_text("To be or not to be", n_chars=50, temperature=0.01))

1/1 [==============================] - 0s 35ms/step
To be or not to be the dujeforful son
and see his kindness oge the s


Not that great I must admit. To generate more convincing text, a common technique consists in only sampling from the top k characters, or only from the smallest set of top characters whose total probability exceeds some threshold(this is called nucleus sampling). Or you could use beam search. We could also try using more GRU layers and more neurons per layer, train for longer, and add some regularization if needed. Moreover, the model is currently incapable of learning patterns longer than defined length which is just 100 characters. You could try making this window larger but it will also make training harder, and even LSTM and GRU cells cannot handle very long sequences. Alternatively, one could use a stateful RNN

### Stateful RNN
For stateless RNNs, at each training iteration, the model starts with a hidden state full of zeros, then it updates this state at each time step and after the last time step, it throws it away as it is not needed anymore. This means the next training batch starts with a state full of initial zeros as well. For Stateful RNNs, the final state at the the last time step of the previous batch is preserved and used as the initial state for the next training batch. This way, the model can learn long term patterns despite only backpropagating through short sequences. However, stateful RNN only makes sense if each input sequence in a batch starts exactly where the corresponding sequence in the previous batch left off. As such, the first thing we need to do to build a stateful RNN is to use sequential and non overlapping input sequences(rather than shuffled and overlapping sequences that's used to train stateless RNNs).

After a stateful model is trained, it will only be possible to use it to make predictions for batches of the same size as were used during training. To avoid this restriction, one should create an identical stateless model and copy the stateful model's weights to this model.

Interestingly, although a CharRNN model is trained to just predict the next character, this seemingly simple task actually requires it to learn some higher level tasks as well.

### Sentiment Analysis

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf

In [ ]:
raw_train_set, raw_valid_set, raw_test_set = tfds.load(
    name="imdb_reviews",
    split=["train[:90%]", "train[90%:]", "test"],
    as_supervised = True
)
tf.random.set_seed(42)
train_set = raw_train_set.shuffle(5000, seed=42).batch(32).prefetch(1)
valid_set = raw_train_set.batch(32).prefetch(1)
test_set = raw_test_set.batch(32).prefetch(1)

In [ ]:
# inspect a few reviews:
for review, label in raw_train_set.take(4):
  print(review.numpy().decode("utf-8"))
  print("Label: ", label.numpy())

This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.
Label:  0
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film was rubbish. The plot developmen

For our task, let us create a TextVectorization layer and adapt it to the training set. We will limit the vocabulary 1000 tokens, including the most frequent 998 words, plus a padding token and a token for unknow words, since it's unlikely that vary rare words will be important for this task, and limiting the vocabulary size will reduce the number of parameters the model needs to learn.

In [ ]:
vocab_size = 1000
text_vec_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.map(lambda reviews, labels: reviews))

In [ ]:
# create a model and train it
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])

history = model.fit(train_set, validation_data=valid_set, epochs=2)

Epoch 1/2
704/704 [==============================] - 1024s 1s/step - loss: 0.6934 - accuracy: 0.5003 - val_loss: 0.6930 - val_accuracy: 0.5012
Epoch 2/2
704/704 [==============================] - 1147s 2s/step - loss: 0.6930 - accuracy: 0.5030 - val_loss: 0.6913 - val_accuracy: 0.5026


From the above result, it can be seen that the model fails to learn anything at all as the accuracy remains close to 50%, this is no better than random chance. This is because the reviews have different lengths so when the TextVectorization layer converts them to sequences of token IDs, it pads the shorter sequences using the padding token(with ID 0) to make them as long as the longest sequence in the batch. As a result, most sequences end with many padding tokens, often dozens or even hundreds of them. Even though we are using a GRU layer which is much better than a SimpleRNN layer, its short term memory is still not great, so when it goes through many padding tokens, it ends up forgetting what the review was about. One solution is to feed the model with batches of equal-length sentences or make the RNN ignore the padding tokens. This can be done using masking

### Masking


In [ ]:
# create a model and train it
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True), # mask zero
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])

history = model.fit(train_set, validation_data=valid_set, epochs=2)

# read the note for more useful info on masking

Another approach that can be used to solve the above issue aside masking would be to feed the model with ragged tensors. This can be done by setting ragged=True when creating the TextVectorization layer, so the input sequences are represented as ragged tensors

In [ ]:
text_vec_layer_ragged = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size, ragged=True
)
text_vec_layer_ragged.adapt(train_set.map(lambda reviews, labels: reviews))
text_vec_layer_ragged([
    "Great Movie!", "This is DiCaprio's best role."
])

<tf.RaggedTensor [[86, 18], [11, 7, 1, 116, 217]]>

In [ ]:
# Comparing the ragged tensor representation with regular tensor representation
# which used padding tokes
text_vec_layer([
    "Great Movie!", "This is DiCaprio's best role."
])

<tf.Tensor: shape=(2, 5), dtype=int64, numpy=
array([[ 86,  18,   0,   0,   0],
       [ 11,   7,   1, 116, 217]])>

### Reusing Pretrained Embeddings and Language Models

Read textbook for more info.

An encoder Decoder network for neural machine translation.
A simple Neural Machine Translation model that translates english sentences to spanish.

The architecture is as follows: English sentences are fed as inputs to the encoder, and the decoder outputs the spanish translations. However, during training, the spanish translations are also used as inputs to the decoder but are shifted back by one step. I.e, during training, the decoder is given the word that it should have output at the previous step, regardless of what it actually output. This is called teacher forcing and it is a technique that significantly speeds up the training and improves the model's performance.for the very first word, the decoder is given the state-of-sequence(SOS) token and the decoder is expected to end the sentence with an end-of-sequence(EOS) token.

Each word is initially represented by its ID, next, an Embedding layer returns the word embedding. These word embeddings are then fed to the encoder and the decoder.

### Bidirectional RNNs
At each time step, a regular recurrent layer only looks at past and present inputs before generating its output, this means it is causal as it cannot look into the future. This makes sense when forecasting time series, or in the decoder of a sequence-to-sequence(seq2seq) model. However, for tasks like text classification, or in the encoder of a seq2seq model, it is often preferable to look ahead at the next words before encoding a given word. Eg, consider the phrases "the right arm", "the right person", and "the right to criticize". to properly encode the word "right", one needs to look ahead.

One solution to this is to run two recurrent layers on the same inputs, one reading the words from left to right and the other reading them from right to left. Then their outputs at each time step are combined, typically by concatenating them. This is what a bidirectional recurrent layer does.

### Beam Search
Read in book - Very useful concept.
Essentially, sometimes, when translating texts, a model may find that it has made a mistake earlier, but can't go back to fix the mistake made, the beam search is a way to give the model a chance to go back and fix its mistakes. How it does this is that it keeps track of a short list of the k most promising sentences(say top 3) and at each decoder step it tries to extend them them by one word, keeping only the k most likely sentences. This parameter k is called the beam width.

check it out on the book for more


### Attention Mechanisms
These are techniques that allow the decoder to focus on the appropriate words (as encoded by the encoder) at each time step. Eg, in the translation of the phrase "I like soccer" to " Me gusta el futbol", at the time step where the decoder needs to output the word "futbol", it will focus it's attention on the word "soccer". This means the path from an input word to its translation is now much shorter, so the short term memory limitations of RNNs have much less impact.


### Read more about them. Very very useful concepts.
One benefit of attention mechanisms is that they make it easier to understand what led the model to produce its output. This is called explainability and can be especially useful when the model makes a mistake. Not only is explainability a tool to debug models, in some applications, it can be a legal requirement. For example, in a system deciding whether or not it should grant one a loan.


#### *Side Info: An inductive bias is an implicit assumption made by the model, due to its architecture. For example, linear models implicitly assume that the data is, well, linear. CNNs implicitly assume that patterns learned in one location will likely be useful in other locations as well. RNNs implicitly assume that the inputs are ordered, and that recent tokens are more important than older ones. The more inductive biases a model has, assuming they are correct, the less training data the model will require. But if the implicit assumptions are wrong, then the model may perform poorly ieven if it is trained on a large dataset.

### Hugging Face

In [ ]:
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 37.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.6 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline

In [ ]:
classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'
Xformers is not installed correctly. If you want to use memory_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [ ]:
result = classifier("The actors were very convicing.")

In [ ]:
result

[{'label': 'POSITIVE', 'score': 0.9979361295700073}]

In [ ]:
classifier([I am from India.", "I am from Iraq.", "I am from Nigeria.])

[{'label': 'NEGATIVE', 'score': 0.9958819150924683}]

### Bias and Fairness

From the output, it suggests that the classifier is biased against Iraqis. this undesirable bias generally comes in large part from the training data itset: in this case, there were plenty of negative sentences related to wars in Iraq in the training data. This bias was then amplified during the fine-tuning process since the model was forced to choose between just two classes: positive or negative. If a neutral class were added when fine-tuning, then the country bias mostly disappears. Also, the training data is not the only source of bias: the model's architecture, the type of loss or regularization used for training, the optimizer, all of these can affect what the model ends up learning. Even a mostly unbiased model can be used in a biased way, much like survey questions can be biased.

Understanding bias in A.I. and mitigating its negative effects is still an area of active research, but one thing is certain: You should pause and think before you rush to deploy a model to production. Ask yourself how the model could do harm, even indirectly. For example, if the model's predictions are used to decide whether or not to give someone a loan, the process should be fair. So make sure you evaluate the model's performance not just on average over the whole test set, but across various subsets as well: for example, you may find that although the model works very well on average, its performance is abysmal for some categories of people. YOu may want to run counterfactual tests: for example, you may want to check that the model's predictions do not change when you simply switch someone's gender. If the model works well on average, it's tempting to push it to production and move on to something else, especially if it's just one component of a much larger system. But in general, if you don't fix such issues, no one else will, and your model may end up doing more harm than good. The solution depends on the problem: it may require rebalancing the dataset, fine-tuning on a different dataset, switching to another pretrained model, tweaking the model's architecture or hyperparameters etc.